# Classical Machine Learning Baseline (CamemBERT-Bio CLS Embeddings + Structured Covariates)

This module extends the classical baseline by incorporating **structured clinical
covariates** alongside frozen CamemBERT-Bio CLS embeddings. It mirrors the multimodal
fine‑tuning setup and uses the **same fixed train/test splits** generated during the
unimodal preprocessing stage (`splits_unimodal/train_seed_*.csv`).

## Included Covariates

All covariates are preprocessed and standardized in the split-generation step:

- **Age**
- **Sex** (binary encoded)
- **Evolution** (ordinal clinical outcome)

These features are concatenated with the CLS embeddings to form a multimodal input
vector for classical ML models.

## Key Features

- **Frozen CamemBERT-Bio encoder**  
  CLS embeddings extracted without updating model weights.

- **Structured covariate integration**  
  Age (standardized), sex, and evolution appended to the embedding vector.

- **Multiple imbalance strategies**  
  - Baseline  
  - Class weights  
  - Random undersampling  
  - Random oversampling  
  - SMOTE  
  - ENN  
  - SMOTEENN  

- **Models evaluated**  
  - Linear SVM (SVC)  

- **Evaluation metrics (F1‑optimal threshold)**  
  - Accuracy  
  - Precision, Recall, F1  
  - Sensitivity, Specificity  
  - **PPV**, **NPV**  
  - ROC‑AUC  
  - AUPRC  
  - **Runtime per model per seed**

- **Reproducibility**  
  Experiments run across **five deterministic seeds** (42–46), identical to the
  multimodal fine‑tuning pipeline.

- **Statistical confidence**  
  All metrics reported with **bootstrap 95% confidence intervals**.

This pipeline provides a rigorous multimodal classical baseline, enabling a fair and
transparent comparison with transformer fine‑tuning when structured clinical
information is available.


In [ ]:
# =============================================================================
# STEP 1 — GENERATE FIXED TRAIN/VAL/TEST SPLITS (70/10/20)
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split
import os

df = pd.read_excel("GBS_FINAL_II.xlsx")
df = df.rename(columns={"Narratif": "text", "gbs_smq_label": "label"})
df["label"] = df["label"].astype(int)

# Covariates
df["age"] = df["Age_numeric"].astype(float)
df["sex"] = df["Sex_numeric"].astype(float)
df["evolution"] = df["Evolution_numeric"].astype(float)

# Normalize age
df["age"] = (df["age"] - df["age"].mean()) / df["age"].std()

# Save directory
splits = "/Users/keneni/Desktop/Bordeaux/Output/splits_unimodal"
os.makedirs(splits, exist_ok=True)

# Deterministic seeds
SPLIT_SEEDS = [42, 43, 44, 45, 46]

# ============================================================
# GENERATE 70/10/20 SPLITS (Train/Val/Test)
# ============================================================
for seed in SPLIT_SEEDS:
    # First split: 70% train, 30% temporary
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df["label"], random_state=seed
    )

    # Second split: 10% val, 20% test (2/3 of temp = 20% of total)
    val_df, test_df = train_test_split(
        temp_df, test_size=2/3, stratify=temp_df["label"], random_state=seed
    )

    # Save all three splits
    train_df.to_csv(f"{splits}/train_seed_{seed}.csv", index=False)
    val_df.to_csv(f"{splits}/val_seed_{seed}.csv", index=False)
    test_df.to_csv(f"{splits}/test_seed_{seed}.csv", index=False)

    print(f"Seed {seed}: Train={len(train_df)} (pos:{train_df['label'].sum()}), "
          f"Val={len(val_df)} (pos:{val_df['label'].sum()}), "
          f"Test={len(test_df)} (pos:{test_df['label'].sum()})")

print(f"Saved 5 deterministic unimodal splits (70/10/20) in: {splits}")


In [ ]:


# =============================================================================
# CamemBERT-Bio + Classical ML + Covariates
# Fixed Multimodal Splits (70/10/20) | All Imbalance Strategies | PPV, NPV, Runtime, CI
# =============================================================================

from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.combine import SMOTEENN
import random
import numpy as np
import torch
import pandas as pd
import os
import warnings
import time
from collections import Counter
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc,
    average_precision_score,
    f1_score
)

warnings.filterwarnings('ignore')

# -------------------------------
# Parameters
# -------------------------------
BASE_SEEDS = [42, 43, 44, 45, 46]

# Fixed splits used by multimodal/fine-tuned pipeline (NOW WITH VALIDATION)
split_dir = "/Users/keneni/Desktop/Bordeaux/Output/splits_unimodal"

# Save directory
save_dir = "/Users/keneni/Desktop/Bordeaux/Output/Final/classical_covariates_all_imbalance"
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cpu")
n_bootstrap = 1000

# -------------------------------
# Embedding function
# -------------------------------
def extract_cls(hf_name, texts):
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModel.from_pretrained(hf_name).to(device)
    model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

# -------------------------------
# Imbalance strategies
# -------------------------------
strategies = {
    "Baseline": ("none", None),
    "ClassWeight": ("none", None),
    "RandomUnderSampler": ("under", RandomUnderSampler()),
    "RandomOverSampler": ("over", RandomOverSampler()),
    "SMOTE": ("over", SMOTE(k_neighbors=3)),
    "ENN": ("under", EditedNearestNeighbours()),
    "SMOTEENN": ("combine", SMOTEENN())
}

# -------------------------------
# Model definition
# -------------------------------
def get_model(model_name, strategy_name, y_train, seed):

    if model_name == "CamemBERT+Cov+LGBM":
        params = {
            "n_estimators": 200,
            "max_depth": 15,
            "deterministic": True,
            "n_jobs": 1,
            "random_state": seed,
            "verbose": -1
        }
        if strategy_name == "ClassWeight":
            neg, pos = np.bincount(y_train)
            params["scale_pos_weight"] = neg / pos
        return LGBMClassifier(**params)

    if model_name == "CamemBERT+Cov+SVC":
        if strategy_name == "ClassWeight":
            return SVC(kernel="linear", probability=True,
                       class_weight="balanced", random_state=seed)
        return SVC(kernel="linear", probability=True, random_state=seed)

# -------------------------------
# Bootstrap CI
# -------------------------------
def bootstrap_ci(values, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    values = np.array(values)
    boot = [np.mean(rng.choice(values, len(values), replace=True))
            for _ in range(n_bootstrap)]
    return np.mean(values), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

# -------------------------------
# Training loop WITH VALIDATION SET
# -------------------------------
results_all = []

for strategy_name, (stype, sampler) in strategies.items():

    print(f"\n{'='*70}")
    print(f"IMBALANCE STRATEGY: {strategy_name}")
    print(f"{'='*70}")

    for seed in BASE_SEEDS:

        print(f"\n--- Seed {seed} ---")

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        # Load fixed multimodal splits (NOW WITH VALIDATION)
        train_df = pd.read_csv(f"{split_dir}/train_seed_{seed}.csv")
        val_df = pd.read_csv(f"{split_dir}/val_seed_{seed}.csv")      # ← NEW
        test_df = pd.read_csv(f"{split_dir}/test_seed_{seed}.csv")

        print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

        # Extract CLS embeddings for ALL sets
        X_train_cls = extract_cls("almanach/camembert-bio-base", list(train_df["text"]))
        X_val_cls = extract_cls("almanach/camembert-bio-base", list(val_df["text"]))    # ← NEW
        X_test_cls = extract_cls("almanach/camembert-bio-base", list(test_df["text"]))

        # Covariates (age already standardized)
        age_train = train_df["age"].values.reshape(-1, 1)
        sex_train = train_df["sex"].values.reshape(-1, 1)
        evo_train = train_df["evolution"].values.reshape(-1, 1)

        age_val = val_df["age"].values.reshape(-1, 1)      # ← NEW
        sex_val = val_df["sex"].values.reshape(-1, 1)      # ← NEW
        evo_val = val_df["evolution"].values.reshape(-1, 1) # ← NEW

        age_test = test_df["age"].values.reshape(-1, 1)
        sex_test = test_df["sex"].values.reshape(-1, 1)
        evo_test = test_df["evolution"].values.reshape(-1, 1)

        # Combine embeddings + covariates for ALL sets
        X_train = np.hstack([X_train_cls, age_train, sex_train, evo_train])
        X_val = np.hstack([X_val_cls, age_val, sex_val, evo_val])    # ← NEW
        X_test = np.hstack([X_test_cls, age_test, sex_test, evo_test])

        y_train = train_df["label"].values
        y_val = val_df["label"].values      # ← NEW
        y_test = test_df["label"].values

        # Apply imbalance strategy (ONLY on training data!)
        if strategy_name in ["Baseline", "ClassWeight"]:
            X_resampled, y_resampled = X_train, y_train
        else:
            sampler.set_params(random_state=seed)
            X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)

        # Train models
        for model_name in ["CamemBERT+Cov+LGBM", "CamemBERT+Cov+SVC"]:

            start_time = time.time()

            clf = get_model(model_name, strategy_name, y_train, seed)
            clf.fit(X_resampled, y_resampled)

            # ============================================================
            # STEP 1: Get predictions on VALIDATION set to tune threshold
            # ============================================================
            val_probs = clf.predict_proba(X_val)[:, 1]

            # Tune threshold on VALIDATION set (NOT test!)
            thresholds = np.linspace(0, 1, 101)
            val_f1_scores = [f1_score(y_val, (val_probs >= th).astype(int))
                             for th in thresholds]
            best_threshold = thresholds[np.argmax(val_f1_scores)]

            print(f"  Best threshold from validation: {best_threshold:.4f}")

            # ============================================================
            # STEP 2: Apply threshold to TEST set (ONLY ONCE)
            # ============================================================
            test_probs = clf.predict_proba(X_test)[:, 1]
            preds = (test_probs >= best_threshold).astype(int)

            # Metrics
            precision, recall, f1, _ = precision_recall_fscore_support(
                y_test, preds, average="binary", zero_division=0
            )
            acc = accuracy_score(y_test, preds)
            cm = confusion_matrix(y_test, preds)
            tn, fp, fn, tp = cm.ravel()

            sensitivity = tp/(tp+fn) if (tp+fn) > 0 else 0
            specificity = tn/(tn+fp) if (tn+fp) > 0 else 0
            ppv = tp/(tp+fp) if (tp+fp) > 0 else 0
            npv = tn/(tn+fn) if (tn+fn) > 0 else 0

            fpr, tpr, _ = roc_curve(y_test, test_probs)
            roc_auc = auc(fpr, tpr)
            auprc = average_precision_score(y_test, test_probs)

            runtime = time.time() - start_time

            results_all.append({
                "strategy": strategy_name,
                "seed": seed,
                "model": model_name,
                "accuracy": acc,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "sensitivity": sensitivity,
                "specificity": specificity,
                "ppv": ppv,
                "npv": npv,
                "roc_auc": roc_auc,
                "auprc": auprc,
                "optimal_threshold": best_threshold,
                "runtime_sec": runtime
            })

            # Display results
            print(f"\n{strategy_name} | {model_name} | Seed {seed}")
            print(f"Optimal threshold (from val): {best_threshold:.4f}")
            print(f"Accuracy: {acc:.4f}")
            print(f"Precision: {precision:.4f}")
            print(f"Recall: {recall:.4f}")
            print(f"F1: {f1:.4f}")
            print(f"Sensitivity: {sensitivity:.4f}")
            print(f"Specificity: {specificity:.4f}")
            print(f"PPV: {ppv:.4f}")
            print(f"NPV: {npv:.4f}")
            print(f"ROC-AUC: {roc_auc:.4f}")
            print(f"AUPRC: {auprc:.4f}")
            print(f"Runtime: {runtime:.2f} sec")

# -------------------------------
# Aggregate results + Bootstrap CI
# -------------------------------
results_df = pd.DataFrame(results_all)

metrics = [
    "accuracy","precision","recall","f1",
    "sensitivity","specificity","ppv","npv",
    "roc_auc","auprc","runtime_sec","optimal_threshold"
]

summary = []

for strategy in results_df["strategy"].unique():
    for model in results_df["model"].unique():

        df_sub = results_df[(results_df["strategy"] == strategy) &
                            (results_df["model"] == model)]

        if len(df_sub) == 0:
            continue

        row = {"strategy": strategy, "model": model}

        for metric in metrics:
            mean_val, ci_low, ci_high = bootstrap_ci(df_sub[metric].values)
            row[metric] = f"{mean_val:.4f} [{ci_low:.4f}-{ci_high:.4f}]"

        summary.append(row)

summary_df = pd.DataFrame(summary)

results_df.to_csv(os.path.join(save_dir, "detailed_covariates_all_imbalance.csv"), index=False)
summary_df.to_csv(os.path.join(save_dir, "summary_covariates_all_imbalance.csv"), index=False)

print("\n=== DONE — Classical ML + Covariates + All Imbalance Strategies (70/10/20 Splits) ===")